<a href="https://colab.research.google.com/github/Maame-Pokuaa77/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile requirements.txt
openai
python-dotenv
pandas
matplotlib

Writing requirements.txt


In [3]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GroqApiKey")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


In [8]:

# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
prompt = ask_llm("What is the premier university in Ghana?")
answer = ask_llm(prompt)
print(answer)

# TODO: Print response.usage as well — how many tokens did your call consume?
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.7,
    max_tokens=500,
)
print(response.usage)

That's correct. The University of Ghana, located in Legon, Accra, is indeed the premier university in Ghana and has a rich history dating back to 1948. As the oldest and largest of the 13 public universities in Ghana, it has established itself as a center of academic excellence, offering a diverse range of programs across various fields, including arts, social sciences, natural sciences, and engineering.

The university's strong reputation is a testament to its commitment to providing high-quality education and research opportunities to its students. With a wide range of undergraduate and graduate programs, the University of Ghana attracts students from all over Ghana and internationally, making it a vibrant and diverse academic community.

The university's location in Accra, the capital city of Ghana, also provides students with access to a wide range of cultural, economic, and social opportunities, making it an ideal place to study and conduct research. Overall, the University of Gha

1. What is the difference between the system and user roles? Give an example of something that belongs in each.

The system role sets up the general structure for how the model is supposed to respond. It makes the model take on a specific role or persona related to a given field, using knowledge from that field to perform the task the user wants. It also defines the standard format the response should follow, along with constraints;for example, that it should stick to credible information and avoid making things up.

Example: "You are a software engineer conducting an interview. Use credible information on how junior software developer interviews are typically conducted, and don't provide any false information."

The user role,is where the actual task is specified and it tells the system exactly what to do in that particular turn.

Example: "Here is a junior software developer's CV , does it fit the job description below?"

2. What is a token, roughly? Why do API providers bill per token rather than per request?

A token is the basic unit of text ; a character, word, or subword  that results from splitting text into fragments an LLM can read, process, and generate.

API providers bill per token rather than per request because pricing based on the request alone would ignore how much work each request actually involves. A single request could contain very few tokens or a huge number of them, and if providers charged a flat rate per request, they'd run at a loss on longer ones. Billing per token lets them charge in proportion to the actual computational cost, since it's the amount of text read and generated and not the number of times a request is sent that drives the cost on their end. This makes  billing per token the more sustainable and fair model.

In [10]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
test_question = "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.

print("Temperature = 0.0")
for i in range(5):
  answer = ask_llm(test_question, temperature=0.0)
  print(f"{i+1}, {answer}")


print("Temperature = 1.2")
for i in range(5):
  answer = ask_llm(test_question, temperature=1.2)
  print(f"{i+1}, {answer}")

Temperature = 0.0
1, Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name plays on the idea of saving being a valuable treasure for market traders.
3. **Sika Saver**: "Sika" is the Ghanaian word for money, so this name incorporates a local touch.
4. **MarketMate Savings**: This name positions the savings product as a trusted companion for market traders.
5. **Kokroko Savings Plan**: "Kokroko" is a Ghanaian term for a collective savings scheme, which could appeal to market traders who are familiar with this concept.
6. **Accra Trader's Fund**: This name emphasizes the product's focus on supporting market traders in Accra.
7. **Suzyo Savings**: "Suzyo" is a Ghanaian term for "save" or "keep", which could make the product more relatable to market traders.

Choose the one that resonates the most with your target audi

What did you observe at each temperature?

Temperature=0.0
The outputs were highly repetitive ;3 of the 5 responses used near-identical structure and even the same top suggestions ("Makola Save"/"Makola Savings" appeared repeatedly, along with "Traders' Treasure," "Sika Saver," "MarketMate Savings"). The model consistently converged on the same most-likely answer each time, since low temperature makes it pick the highest-probability tokens rather than sampling more broadly.

Temperature=1.2
No two responses were the same. Every run produced different name suggestions, different explanations, and sometimes different structures (e.g., some included local-language terms like "SusuBox" ,"Sua" that never appeared at temperature 0). The answers were more creative and varied.

For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?



For the loan decision-support system, temperature 0.0 is the appropriate regime. If the same applicant's letter produced a different summary, different extracted data, or a different recommendation each time it was run, the system would be unreliable and untrustworthy from the loan officer's point of view.The officer needs to know that rerunning the same input gives the same output, especially since this is a decision-support tool feeding into real financial decisions. Low temperature minimizes this inconsistency and keeps outputs deterministic, factual, and reproducible, which matters far more here than creativity does.This kind of task  involves summarization, extraction and recommendation, lower hallucination risk and consistency is the priority thus, temperature = 0.0 is the right choice.